# Ragrails — Ingestion

This notebook covers all three ingestion methods:

- `scrape()` — web pages and websites
- `parse()` — local documents (PDF, DOCX, CSV, and more)
- `fetch()` — REST API responses

Each method converts its source into clean markdown files ready for chunking.

## Install

In [ ]:
# Base install covers document and API ingestion
# %pip install ragrails

# URL ingestion needs the url extra
# %pip install "ragrails[url]"

In [ ]:
from ragrails import RagRails

rag = RagRails()

---

## URL Ingestion

Use `scrape()` to pull web pages into markdown.

`crawl4ai` uses Playwright to load pages. Run `setup_url()` once per environment before scraping.

In [ ]:
# Run once per environment after installing ragrails[url]
rag.setup_url()

### Scrape exact URLs

Use `mode="each"` when you only want the pages you list.

In [ ]:
result = rag.scrape(
    url=[
        "https://example.com/about",
        "https://example.com/pricing",
    ],
    mode="each",
    output_dir="files/output/web_crawled",
)

print("Pages scraped:", result.pages)
print("Pages failed:", result.failed)
print("Output files:", result.files)
print("Errors:", result.errors)

### Crawl a full website

Use `mode="full"` to follow links and crawl an entire site.

In [ ]:
result = rag.scrape(
    url="https://example.com",
    mode="full",
    output_dir="files/output/web_crawled",
    max_depth=3,
    max_pages=200,
)

print("Pages scraped:", result.pages)
print("Pages failed:", result.failed)
print("DLQ path:", result.dlq_path)
print("Output files:", result.files)

### Retry failed URLs

Failed pages are written to a dead-letter queue (DLQ). Use `retry_scrape()` to attempt them again.

In [ ]:
if result.failed:
    retry = rag.retry_scrape(
        result.dlq_path,
        max_attempts=3,
    )
    print("Retried pages scraped:", retry.pages)
    print("Still failed:", retry.failed)
    print("Errors:", retry.errors)

---

## Document Ingestion

Use `parse()` to convert local files into markdown. No extra install needed — included in the base package.

Supported formats: `.pdf`, `.docx`, `.csv`, `.xlsx`, `.pptx`, `.html`, `.md`, `.epub`, `.txt`, and more.

### Parse a folder

Ragrails discovers all supported files in the folder automatically.

In [ ]:
result = rag.parse(
    folder="files/input",
    output_dir="files/output/docs",
)

print("Documents parsed:", result.documents)
print("Documents failed:", result.failed)
print("Output files:", result.files)
print("Errors:", result.errors)

### Parse specific files

In [ ]:
result = rag.parse(
    files=["guide.pdf", "pricing.csv"],
    input_dir="files/input",
    output_dir="files/output/docs",
)

print("Documents parsed:", result.documents)
print("Output files:", result.files)

### Parse with custom metadata

Pass dicts to control the title and description written into frontmatter.

In [ ]:
result = rag.parse(
    files=[
        {
            "filename": "guide.pdf",
            "title": "Product Guide",
            "description": "Internal product guide.",
        },
        {
            "filename": "pricing.csv",
            "title": "Pricing Table",
            "description": "Current product pricing.",
        },
    ],
    input_dir="files/input",
    output_dir="files/output/docs",
)

print("Documents parsed:", result.documents)
print("Output files:", result.files)

### Error handling

In [ ]:
result = rag.parse(
    folder="files/input",
    output_dir="files/output/docs",
)

if result.failed:
    for error in result.errors:
        print("Error:", error)

---

## API Ingestion

Use `fetch()` to pull REST API responses into markdown. No extra install needed.

### Basic GET request

In [ ]:
result = rag.fetch(
    url="https://api.example.com/v1/products",
    title="Products",
    description="Product catalog from the public API.",
    output_dir="files/output/api",
)

print("Pages fetched:", result.pages)
print("Items fetched:", result.items)
print("Output files:", result.files)
print("Errors:", result.errors)

### Request with headers and query params

In [ ]:
result = rag.fetch(
    url="https://api.example.com/v1/products",
    method="GET",
    headers={"Authorization": "Bearer <token>"},
    params={"limit": 100},
    title="Products",
    output_dir="files/output/api",
)

print("Pages fetched:", result.pages)
print("Items fetched:", result.items)

### POST request with body

In [ ]:
result = rag.fetch(
    url="https://api.example.com/v1/search",
    method="POST",
    headers={"Authorization": "Bearer <token>"},
    body={"query": "payments"},
    title="Search Results",
    output_dir="files/output/api",
)

print("Pages fetched:", result.pages)
print("Items fetched:", result.items)

### Paginated API

In [ ]:
result = rag.fetch(
    url="https://api.example.com/v1/products",
    title="Products",
    pagination={
        "type": "page",
        "param": "page",
        "size_param": "limit",
        "size": 100,
    },
    max_pages=20,
    output_dir="files/output/api",
)

print("Pages fetched:", result.pages)
print("Total items:", result.items)

### Error handling

In [ ]:
result = rag.fetch(
    url="https://api.example.com/v1/products",
    title="Products",
    output_dir="files/output/api",
)

if result.failed:
    for error in result.errors:
        print("Error:", error)